# FIFA WORLD CUP DATA VALIDATION

In [35]:
import pandas as pd
from pathlib import Path

# Relative paths (same style as notebooks 01 and 02)
RAW_FOLDER = Path("../Fifa worldcup data/row")
CLEANED_FOLDER = Path("../Fifa worldcup data/cleaned")
VALIDATION_FOLDER = Path("../Fifa worldcup data/validation")
VALIDATION_FOLDER.mkdir(parents=True, exist_ok=True)

# 1. Load all 13 tables

In [36]:
FILES = [
    "Confederations",
    "Federations",
    "Teams",
    "Players",
    "Tournaments",
    "Stadiums",
    "Matches",
    "Team_Appearances",
    "Player_Appearances",
    "Goals",
    "Bookings",
    "Substitutions",
    "Match_Attendance",
]

missing_files = [f for f in FILES if not (CLEANED_FOLDER / f"{f}.csv").exists()]
if missing_files:
    raise FileNotFoundError(f"Missing cleaned files: {missing_files}")

tables = {f: pd.read_csv(CLEANED_FOLDER / f"{f}.csv", low_memory=False) for f in FILES}

print("=" * 60)
print("FIFA WORLD CUP DATA VALIDATION")
print("=" * 60)
print(f"\nTables loaded: {len(tables)}/{len(FILES)}")

FIFA WORLD CUP DATA VALIDATION

Tables loaded: 13/13


# 2. Check helper + total records

In [37]:
results = []

def check(name, condition, details, severity="FAIL"):
    """severity='FAIL' breaks the overall status, 'WARN' only reports."""
    status = "PASS" if condition else severity
    results.append({"Test": name, "Status": status, "Details": details})
    symbol = {"PASS": "✅", "FAIL": "❌", "WARN": "⚠️"}[status]
    print(f"{symbol} {name}: {details}")

total_records = sum(len(df) for df in tables.values())
check("20,000+ records", total_records >= 20000, f"{total_records:,} total records")

✅ 20,000+ records: 69,309 total records


# 3. Primary keys (single and composite)

In [38]:
primary_keys = {
    "Confederations": ["confederationid"],
    "Federations": ["federationid"],
    "Teams": ["teamid"],
    "Players": ["playerid"],
    "Tournaments": ["tournamentid"],
    "Stadiums": ["stadiumid"],
    "Matches": ["matchid"],
    "Team_Appearances": ["matchid", "teamid"],     #checking composite primary key
    "Player_Appearances": ["matchid", "playerid"],  #checking composite primary key
    "Goals": ["goalid"],
    "Bookings": ["bookingid"],
    "Substitutions": ["substitutionid"],
    "Match_Attendance": ["attendanceid"],
}

for table, pk in primary_keys.items():
    df = tables[table]
    missing = int(df[pk].isna().any(axis=1).sum())
    duplicates = int(df.duplicated(subset=pk).sum())
    check(
        f"Primary Key - {table} ({', '.join(pk)})",
        missing == 0 and duplicates == 0,
        f"missing={missing}, duplicates={duplicates}",
    )

✅ Primary Key - Confederations (confederationid): missing=0, duplicates=0
✅ Primary Key - Federations (federationid): missing=0, duplicates=0
✅ Primary Key - Teams (teamid): missing=0, duplicates=0
✅ Primary Key - Players (playerid): missing=0, duplicates=0
✅ Primary Key - Tournaments (tournamentid): missing=0, duplicates=0
✅ Primary Key - Stadiums (stadiumid): missing=0, duplicates=0
✅ Primary Key - Matches (matchid): missing=0, duplicates=0
✅ Primary Key - Team_Appearances (matchid, teamid): missing=0, duplicates=0
✅ Primary Key - Player_Appearances (matchid, playerid): missing=0, duplicates=0
✅ Primary Key - Goals (goalid): missing=0, duplicates=0
✅ Primary Key - Bookings (bookingid): missing=0, duplicates=0
✅ Primary Key - Substitutions (substitutionid): missing=0, duplicates=0
✅ Primary Key - Match_Attendance (attendanceid): missing=0, duplicates=0


# 4. Foreign keys
NaN values are ignored on purpose (e.g. `winner_team_id` can be empty for tournaments without a winner).

In [39]:
foreign_keys = [
    ("Teams", "federationid", "Federations", "federationid"),
    ("Teams", "confederationid", "Confederations", "confederationid"),
    ("Tournaments", "winnerteamid", "Teams", "teamid"),
    ("Matches", "tournamentid", "Tournaments", "tournamentid"),
    ("Matches", "stadiumid", "Stadiums", "stadiumid"),
    ("Matches", "hometeamid", "Teams", "teamid"),
    ("Matches", "awayteamid", "Teams", "teamid"),
    ("Team_Appearances", "matchid", "Matches", "matchid"),
    ("Team_Appearances", "teamid", "Teams", "teamid"),
    ("Team_Appearances", "opponentid", "Teams", "teamid"),
    ("Player_Appearances", "matchid", "Matches", "matchid"),
    ("Player_Appearances", "teamid", "Teams", "teamid"),
    ("Player_Appearances", "playerid", "Players", "playerid"),
    ("Goals", "matchid", "Matches", "matchid"),
    ("Goals", "teamid", "Teams", "teamid"),
    ("Goals", "playerid", "Players", "playerid"),
    ("Goals", "playerteamid", "Teams", "teamid"),
    ("Bookings", "matchid", "Matches", "matchid"),
    ("Bookings", "teamid", "Teams", "teamid"),
    ("Bookings", "playerid", "Players", "playerid"),
    ("Substitutions", "matchid", "Matches", "matchid"),
    ("Substitutions", "teamid", "Teams", "teamid"),
    ("Substitutions", "playerid", "Players", "playerid"),
    ("Match_Attendance", "matchid", "Matches", "matchid"),
]
for child_table, child_col, parent_table, parent_col in foreign_keys:
    child_values = set(tables[child_table][child_col].dropna())
    parent_values = set(tables[parent_table][parent_col].dropna())
    orphans = len(child_values - parent_values)
    check(f"FK {child_table}.{child_col}", orphans == 0, f"orphan values={orphans}")

✅ FK Teams.federationid: orphan values=0
✅ FK Teams.confederationid: orphan values=0
✅ FK Tournaments.winnerteamid: orphan values=0
✅ FK Matches.tournamentid: orphan values=0
✅ FK Matches.stadiumid: orphan values=0
✅ FK Matches.hometeamid: orphan values=0
✅ FK Matches.awayteamid: orphan values=0
✅ FK Team_Appearances.matchid: orphan values=0
✅ FK Team_Appearances.teamid: orphan values=0
✅ FK Team_Appearances.opponentid: orphan values=0
✅ FK Player_Appearances.matchid: orphan values=0
✅ FK Player_Appearances.teamid: orphan values=0
✅ FK Player_Appearances.playerid: orphan values=0
✅ FK Goals.matchid: orphan values=0
✅ FK Goals.teamid: orphan values=0
✅ FK Goals.playerid: orphan values=0
✅ FK Goals.playerteamid: orphan values=0
✅ FK Bookings.matchid: orphan values=0
✅ FK Bookings.teamid: orphan values=0
✅ FK Bookings.playerid: orphan values=0
✅ FK Substitutions.matchid: orphan values=0
✅ FK Substitutions.teamid: orphan values=0
✅ FK Substitutions.playerid: orphan values=0
✅ FK Match_Atte

# 5. Match validation

In [40]:
matches = tables["Matches"]

# Expected count comes from the raw file, not a hard-coded number
raw_matches_path = RAW_FOLDER / "Matches.csv"

if raw_matches_path.exists():
    raw_count = len(
        pd.read_csv(raw_matches_path)
        .dropna(how="all")
        .drop_duplicates()
    )

    check(
        "Match count = raw file",
        len(matches) == raw_count,
        f"cleaned={len(matches):,}, raw={raw_count:,}"
    )
else:
    check(
        "Match count",
        len(matches) > 0,
        f"{len(matches):,} matches (raw file not found)",
        severity="WARN"
    )

check(
    "Home team != Away team",
    (matches["hometeamid"] != matches["awayteamid"]).all(),
    "No match has the same team on both sides"
)

check(
    "No negative scores",
    (
        (matches["homescore"] >= 0)
        & (matches["awayscore"] >= 0)
    ).all(),
    "All scores are >= 0"
)

check(
    "No missing scores",
    matches[["homescore", "awayscore"]].notna().all().all(),
    "home/away scores are all filled"
)

✅ Match count = raw file: cleaned=1,352, raw=1,352
✅ Home team != Away team: No match has the same team on both sides
✅ No negative scores: All scores are >= 0
✅ No missing scores: home/away scores are all filled


# 6. Team appearances

In [41]:
team_apps = tables["Team_Appearances"]

wrong_counts = int(
    (team_apps.groupby("matchid").size() != 2).sum()
)
check(
    "2 team appearances per match",
    wrong_counts == 0,
    f"{wrong_counts} matches have incorrect number of teams"
)
missing_matches = set(matches["matchid"]) - set(team_apps["matchid"])
check(
    "Every match has team appearances",
    len(missing_matches) == 0,
    f"{len(missing_matches)} matches without appearances"
)
# Goals in Team_Appearances should equal the final score in Matches
goals_per_match = (
    team_apps
    .groupby("matchid")["goalsfor"]
    .sum()
)
expected = (
    matches
    .set_index("matchid")[["homescore", "awayscore"]]
    .sum(axis=1)
)
diff = (goals_per_match - expected).dropna()
mismatch = int((diff != 0).sum())

check(
    "Goals Team_Appearances vs scores Matches",
    mismatch == 0,
    f"{mismatch} matches differ",
    severity="WARN"
)

✅ 2 team appearances per match: 0 matches have incorrect number of teams
✅ Every match has team appearances: 0 matches without appearances
✅ Goals Team_Appearances vs scores Matches: 0 matches differ


# 7. Logical checks

In [42]:
# Match dates inside the tournament window
t = tables["Tournaments"][
    ["tournamentid", "startdate", "enddate"]
].copy()

t["startdate"] = pd.to_datetime(t["startdate"], errors="coerce")
t["enddate"] = pd.to_datetime(t["enddate"], errors="coerce")

m = matches[
    ["matchid", "tournamentid", "matchdate"]
].copy()

m["matchdate"] = pd.to_datetime(
    m["matchdate"],
    errors="coerce"
)

m = m.merge(
    t,
    on="tournamentid",
    how="left"
)

bad_dates = int(
    (
        (m["matchdate"] < m["startdate"])
        | (m["matchdate"] > m["enddate"])
    ).sum()
)

check(
    "Match date within tournament dates",
    bad_dates == 0,
    f"{bad_dates} matches outside range"
)


# Attendance vs stadium capacity
# Historic data may legitimately exceed it -> WARN only

att = (
    tables["Match_Attendance"]
    .merge(
        matches[["matchid", "stadiumid"]],
        on="matchid",
        how="left"
    )
    .merge(
        tables["Stadiums"][
            ["stadiumid", "stadiumcapacity"]
        ],
        on="stadiumid",
        how="left"
    )
)

over = int(
    (att["attendance"] > att["stadiumcapacity"]).sum()
)

check(
    "Attendance <= stadium capacity",
    over == 0,
    f"{over} matches exceed capacity",
    severity="WARN"
)

✅ Match date within tournament dates: 0 matches outside range
⚠️ Attendance <= stadium capacity: 99 matches exceed capacity


# 8. Duplicate rows

In [43]:
duplicate_tables = [f"{n}: {df.duplicated().sum()}" for n, df in tables.items() if df.duplicated().sum() > 0]
check("No duplicate rows", not duplicate_tables,
      "No exact duplicate rows" if not duplicate_tables else ", ".join(duplicate_tables))

✅ No duplicate rows: No exact duplicate rows


# 9. Final result

In [44]:
results_df = pd.DataFrame(results)
failures = int((results_df["Status"] == "FAIL").sum())
warnings = int((results_df["Status"] == "WARN").sum())

print("\n" + "=" * 60)
if failures == 0:
    print("🎉 OVERALL STATUS: PASS" + (f" (with {warnings} warnings)" if warnings else ""))
else:
    print(f"⚠️ OVERALL STATUS: FAIL — failed tests: {failures}, warnings: {warnings}")
print("=" * 60)

report_path = VALIDATION_FOLDER / "validation_report.csv"
results_df.to_csv(report_path, index=False)
print(f"\n📄 Report saved to: {report_path.resolve()}")


🎉 OVERALL STATUS: PASS (with 1 warnings)

📄 Report saved to: C:\Users\shima\Desktop\FIFA-World-Cup-Data-Analytics\Fifa worldcup data\validation\validation_report.csv
